In [1]:
import numpy as np
import pandas as pd

In [2]:
STUDENT_NAME = "Zainier Manombaga"
STUDENT_ID = 3683

print(f"Student: {STUDENT_NAME}")
print(f"ID: {STUDENT_ID}")

Student: Zainier Manombaga
ID: 3683


In [3]:
# Make sure the CSV file is in the same folder
df = pd.read_csv("Most Popular Programming Languages.csv")

# Convert Month column to datetime
df['Month'] = pd.to_datetime(df['Month'])

print("\nDataset Loaded Successfully!")
print(df.head())


Dataset Loaded Successfully!
       Month  Python Worldwide(%)  JavaScript Worldwide(%)  Java Worldwide(%)  \
0 2004-01-01                   30                       98                 96   
1 2004-02-01                   29                       98                 97   
2 2004-03-01                   28                      100                100   
3 2004-04-01                   28                       98                 97   
4 2004-05-01                   28                       91                 99   

   C# Worldwide(%)  PhP Worldwide(%)  Flutter Worldwide(%)  \
0               76               100                     6   
1               86                99                     6   
2               87                97                     5   
3               89               100                     6   
4               84                92                     6   

   React Worldwide(%)  Swift Worldwide(%)  TypeScript Worldwide(%)  \
0                   1                   

In [4]:
languages = sorted(df.columns[1:])  # exclude 'Month'

print("\nAvailable Languages:")
print(languages)


Available Languages:
['C# Worldwide(%)', 'Flutter Worldwide(%)', 'Java Worldwide(%)', 'JavaScript Worldwide(%)', 'Matlab Worldwide(%)', 'PhP Worldwide(%)', 'Python Worldwide(%)', 'React Worldwide(%)', 'Swift Worldwide(%)', 'TypeScript Worldwide(%)']


In [5]:
last_three_digits = STUDENT_ID % 1000
language_index = last_three_digits % len(languages)

assigned_language = languages[language_index]

print("\nAssigned Language:", assigned_language)


Assigned Language: JavaScript Worldwide(%)


In [6]:
# EXERCISE 1: GROWTH ANALYSIS

print("\n================ EXERCISE 1 ================\n")

# Extract data
lang_data = df[['Month', assigned_language]].copy()
lang_data.columns = ['Month', 'Popularity']

# Growth rate
lang_data['Growth_Rate'] = lang_data['Popularity'].pct_change() * 100

# Rolling statistics (3-month)
lang_data['Moving_Avg'] = lang_data['Popularity'].rolling(window=3).mean()
lang_data['Moving_STD'] = lang_data['Popularity'].rolling(window=3).std()

# Phase classification
conditions = [
    lang_data['Growth_Rate'] > 5,
    lang_data['Growth_Rate'] < -5
]

choices = ['Growth', 'Decline']

lang_data['Phase'] = np.select(conditions, choices, default='Stable')

# Summary statistics
summary = lang_data['Popularity'].describe()

# Growth metrics
initial_value = lang_data['Popularity'].iloc[0]
final_value = lang_data['Popularity'].iloc[-1]

overall_growth = ((final_value - initial_value) / initial_value) * 100

phase_counts = lang_data['Phase'].value_counts()

# OUTPUT
print(lang_data.head())

print("\nSummary Statistics:\n", summary)

print("\nPhase Counts:\n", phase_counts)

print("\nKey Metrics:")
print("Initial:", initial_value)
print("Final:", final_value)
print("Overall Growth %:", overall_growth)


================ EXERCISE 1 ================

       Month  Popularity  Growth_Rate  Moving_Avg  Moving_STD    Phase
0 2004-01-01          98          NaN         NaN         NaN   Stable
1 2004-02-01          98     0.000000         NaN         NaN   Stable
2 2004-03-01         100     2.040816   98.666667    1.154701   Stable
3 2004-04-01          98    -2.000000   98.666667    1.154701   Stable
4 2004-05-01          91    -7.142857   96.333333    4.725816  Decline

Summary Statistics:
 count    249.000000
mean      43.963855
std       16.024512
min       24.000000
25%       34.000000
50%       37.000000
75%       48.000000
max      100.000000
Name: Popularity, dtype: float64

Phase Counts:
 Phase
Stable     149
Decline     59
Growth      41
Name: count, dtype: int64

Key Metrics:
Initial: 98
Final: 31
Overall Growth %: -68.36734693877551


In [7]:
# EXERCISE 2: LIFECYCLE CLASSIFICATION

print("\n================ EXERCISE 2 ================\n")

# Growth rate stats
mean_growth = np.mean(lang_data['Growth_Rate'].dropna())
std_growth = np.std(lang_data['Growth_Rate'].dropna())

# Lifecycle rules
conditions = [
    (lang_data['Growth_Rate'] > 0) & (lang_data['Growth_Rate'] < mean_growth),
    (lang_data['Growth_Rate'] > mean_growth),
    (abs(lang_data['Growth_Rate']) <= 1),
    (lang_data['Growth_Rate'] < -std_growth)
]

choices = ['Introduction', 'Growth', 'Maturity', 'Decline']

lang_data['Lifecycle_Phase'] = np.select(conditions, choices, default='Stable')

# Summary
lifecycle_counts = lang_data['Lifecycle_Phase'].value_counts()

print(lang_data[['Month', 'Popularity', 'Growth_Rate', 'Lifecycle_Phase']].head())

print("\nLifecycle Counts:\n", lifecycle_counts)

dominant_phase = lifecycle_counts.idxmax()

print("\nDominant Phase:", dominant_phase)


================ EXERCISE 2 ================

       Month  Popularity  Growth_Rate Lifecycle_Phase
0 2004-01-01          98          NaN          Stable
1 2004-02-01          98     0.000000          Growth
2 2004-03-01         100     2.040816          Growth
3 2004-04-01          98    -2.000000          Stable
4 2004-05-01          91    -7.142857         Decline

Lifecycle Counts:
 Lifecycle_Phase
Growth     137
Stable      79
Decline     33
Name: count, dtype: int64

Dominant Phase: Growth


In [8]:
# EXERCISE 3: COMPARATIVE ANALYSIS
print("\n================ EXERCISE 3 ================\n")

# Select second language (next alphabetically)
second_index = (language_index + 1) % len(languages)
second_language = languages[second_index]

print("Language A:", assigned_language)
print("Language B:", second_language)

# Extract both
df_compare = df[['Month', assigned_language, second_language]].copy()
df_compare.columns = ['Month', 'LangA', 'LangB']

# Stats
mean_A = df_compare['LangA'].mean()
mean_B = df_compare['LangB'].mean()

std_A = df_compare['LangA'].std()
std_B = df_compare['LangB'].std()

# Correlation
correlation = np.corrcoef(df_compare['LangA'], df_compare['LangB'])[0,1]

# Dominance ratio
dominance_count = np.sum(df_compare['LangA'] > df_compare['LangB'])
total = len(df_compare)

dominance_ratio = (dominance_count / total) * 100

# Relative Performance Index (simple version)
rpi = mean_A / mean_B

# Cross-over points
cross_over = df_compare[df_compare['LangA'] > df_compare['LangB']]

# Summary DataFrame
summary_df = pd.DataFrame({
    'Mean_A': [mean_A],
    'Mean_B': [mean_B],
    'Std_A': [std_A],
    'Std_B': [std_B],
    'Correlation': [correlation],
    'Dominance_Ratio (%)': [dominance_ratio],
    'RPI': [rpi]
})

# OUTPUT
print("\nComparison Summary:\n", summary_df)

print("\nCross-over Points:\n", cross_over.head())


================ EXERCISE 3 ================

Language A: JavaScript Worldwide(%)
Language B: Matlab Worldwide(%)

Comparison Summary:
       Mean_A     Mean_B      Std_A     Std_B  Correlation  \
0  43.963855  59.289157  16.024512  13.28217      0.52026   

   Dominance_Ratio (%)       RPI  
0            14.859438  0.741516  

Cross-over Points:
        Month  LangA  LangB
0 2004-01-01     98     78
1 2004-02-01     98     91
2 2004-03-01    100     99
3 2004-04-01     98     95
4 2004-05-01     91     86
